In [1]:
#Patrones y factores asociados a la consumación de extorsiones registradas por Seguridad 
#Ciudadana en Nezahualcóyotl mediante minería de datos

In [1]:
#Bloque 0. Instalación de librerías
#Ejecuta esto una sola vez
!pip install pandas numpy openpyxl scikit-learn matplotlib mlxtend kmodes


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [48]:
#Bloque 1. Cargar librerías y archivo
#Fase KDD: Selección de datos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import unicodedata

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

from kmodes.kmodes import KModes


ruta = "Encuesta-Modificada.xlsx"
df = pd.read_excel(ruta)

print("Dimensiones:", df.shape)
print("\nColumnas originales:")
print(df.columns.tolist())

df.head()

Dimensiones: (66, 15)

Columnas originales:
['Marca temporal', 'Carrera', 'Semestre que cursas', '¿Cuál es la Inteligencia Artificial que usas en tu día a día?', 'Edad', '¿Cuál es el uso principal que le das a la Inteligencia Artificial que seleccionaste?', '¿Con qué frecuencia usas IA para tareas escolares?', ' Intento resolver la tarea por mi cuenta antes de usar IA.  ', 'Uso IA cuando no entiendo un tema explicado en clase.  ', 'Cuando realizas trabajos escolares con apoyo de la IA ¿Revisas y modificas lo que genera la IA antes de entregarlo? ', 'Siento que cada vez dependo más de la IA. ', 'Si no tuviera acceso a IA, mi rendimiento bajaría.  ', 'Prefiero usar IA que investigar en libros o artículos. ', 'Considero que podría dejar de usar IA sin problema. ', 'En una escala del 1 al 5, ¿qué tan dependiente te consideras de la IA?\nDonde 1 es nada dependiente y 5 es totalmente dependiente ']


,Marca temporal,Carrera,Semestre que cursas,¿Cuál es la Inteligencia Artificial que usas en tu día a día?,Edad,¿Cuál es el uso principal que le das a la Inteligencia Artificial que seleccionaste?,¿Con qué frecuencia usas IA para tareas escolares?,Intento resolver la tarea por mi cuenta antes de usar IA.,Uso IA cuando no entiendo un tema explicado en clase.,Cuando realizas trabajos escolares con apoyo de la IA ¿Revisas y modificas lo que genera la IA antes de entregarlo?,Siento que cada vez dependo más de la IA.,"Si no tuviera acceso a IA, mi rendimiento bajaría.",Prefiero usar IA que investigar en libros o artículos.,Considero que podría dejar de usar IA sin problema.,"En una escala del 1 al 5, ¿qué tan dependiente te consideras de la IA?\nDonde 1 es nada dependiente y 5 es totalmente dependiente"
0,2026-02-27 21:20:39.590,Ingenieria en Sistemas Inteligentes,6,Chat GPT,Mayor que 20,Uso general,A veces,A veces,A veces,Si,No,No,Si,Si,3.0
1,2026-02-27 21:20:44.168,Ingenieria en Sistemas Inteligentes,6,Chat GPT,Mayor que 20,Uso general,Casi siempre,A veces,A veces,Si,Si,No,Si,Si,3.0
2,2026-02-27 21:21:05.880,Ingenieria en Sistemas Inteligentes,6,Claude,Mayor que 20,Modo agente,Rara vez,Siempre,A veces,Si,No,No,Si,Si,1.0
3,2026-02-27 21:21:50.365,Ingenieria en Sistemas Inteligentes,6,Chat GPT,Mayor que 20,Uso general,Casi siempre,A veces,Casi siempre,Si,Si,Si,Si,Si,4.0
4,2026-02-27 21:21:56.632,Ingenieria en Sistemas Inteligentes,6,Gemini,Mayor que 20,Uso general,Siempre,A veces,A veces,Si,Si,Si,Si,Si,3.0


In [49]:
# BLOQUE 2 Normalizar nombres de columnas
#Aquí está la corrección clave
#Este bloque sustituye el renombrado manual.
#Así evitamos errores por espacios, acentos o paréntesis.
#Fase KDD: Selección + preparación inicial

def normalizar_nombre_columna(col):
    col = str(col).strip().upper()
    col = ''.join(
        c for c in unicodedata.normalize('NFD', col)
        if unicodedata.category(c) != 'Mn'
    )
    col = re.sub(r'[^A-Z0-9]+', '_', col)
    col = re.sub(r'_+', '_', col).strip('_')
    return col

df.columns = [normalizar_nombre_columna(c) for c in df.columns]

print("Columnas normalizadas:")
print(df.columns.tolist())

Columnas normalizadas:
['MARCA_TEMPORAL', 'CARRERA', 'SEMESTRE_QUE_CURSAS', 'CUAL_ES_LA_INTELIGENCIA_ARTIFICIAL_QUE_USAS_EN_TU_DIA_A_DIA', 'EDAD', 'CUAL_ES_EL_USO_PRINCIPAL_QUE_LE_DAS_A_LA_INTELIGENCIA_ARTIFICIAL_QUE_SELECCIONASTE', 'CON_QUE_FRECUENCIA_USAS_IA_PARA_TAREAS_ESCOLARES', 'INTENTO_RESOLVER_LA_TAREA_POR_MI_CUENTA_ANTES_DE_USAR_IA', 'USO_IA_CUANDO_NO_ENTIENDO_UN_TEMA_EXPLICADO_EN_CLASE', 'CUANDO_REALIZAS_TRABAJOS_ESCOLARES_CON_APOYO_DE_LA_IA_REVISAS_Y_MODIFICAS_LO_QUE_GENERA_LA_IA_ANTES_DE_ENTREGARLO', 'SIENTO_QUE_CADA_VEZ_DEPENDO_MAS_DE_LA_IA', 'SI_NO_TUVIERA_ACCESO_A_IA_MI_RENDIMIENTO_BAJARIA', 'PREFIERO_USAR_IA_QUE_INVESTIGAR_EN_LIBROS_O_ARTICULOS', 'CONSIDERO_QUE_PODRIA_DEJAR_DE_USAR_IA_SIN_PROBLEMA', 'EN_UNA_ESCALA_DEL_1_AL_5_QUE_TAN_DEPENDIENTE_TE_CONSIDERAS_DE_LA_IA_DONDE_1_ES_NADA_DEPENDIENTE_Y_5_ES_TOTALMENTE_DEPENDIENTE']


In [6]:
#Bloque 3. Auditoría rápida de calidad
#Fase KDD: Preprocesamiento
#Esto te dice cuántos valores únicos tiene cada variable, cuántos nulos hay y ejemplos de categorías.

print("Nulos por columna:\n")
print(df.isna().sum())

print("\nValores únicos por columna:\n")
for col in df.columns:
    print(f"\n--- {col} ---")
    print("n_unique:", df[col].nunique(dropna=False))
    print(df[col].astype(str).value_counts(dropna=False).head(10))

Nulos por columna:

MARCA_TEMPORAL                                                                                                                   0
CARRERA                                                                                                                          0
SEMESTRE_QUE_CURSAS                                                                                                              0
CUAL_ES_LA_INTELIGENCIA_ARTIFICIAL_QUE_USAS_EN_TU_DIA_A_DIA                                                                      0
EDAD                                                                                                                             0
CUAL_ES_EL_USO_PRINCIPAL_QUE_LE_DAS_A_LA_INTELIGENCIA_ARTIFICIAL_QUE_SELECCIONASTE                                               0
CON_QUE_FRECUENCIA_USAS_IA_PARA_TAREAS_ESCOLARES                                                                                 0
INTENTO_RESOLVER_LA_TAREA_POR_MI_CUENTA_ANTES_DE_USAR_IA       

In [50]:
#Bloque 4. Funciones de limpieza
#Fase KDD: Preprocesamiento
#Aquí limpiamos texto, acentos, espacios y estandarizamos categorías.

def limpiar_texto(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().upper()
    x = re.sub(r"\s+", " ", x)
    return x

def quitar_acentos(texto):
    if pd.isna(texto):
        return np.nan
    texto = str(texto)
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    return texto



cols_texto_esperadas = [
    "MARCA_TEMPORAL",
    "CARRERA",
    "SEMESTRE_QUE_CURSAS",
    "CUAL_ES_LA_INTELIGENCIA_ARTIFICIAL_QUE_USAS_EN_TU_DIA_A_DIA",
    "EDAD",
    "CUAL_ES_EL_USO_PRINCIPAL_QUE_LE_DAS_A_LA_INTELIGENCIA_ARTIFICIAL_QUE_SELECCIONASTE",
    "CON_QUE_FRECUENCIA_USAS_IA_PARA_TAREAS_ESCOLARES",
    "INTENTO_RESOLVER_LA_TAREA_POR_MI_CUENTA_ANTES_DE_USAR_IA",
    "USO_IA_CUANDO_NO_ENTIENDO_UN_TEMA_EXPLICADO_EN_CLASE",
    "CUANDO_REALIZAS_TRABAJOS_ESCOLARES_CON_APOYO_DE_LA_IA_REVISAS_Y_MODIFICAS_LO_QUE_GENERA_LA_IA_ANTES_DE_ENTREGARLO",
    "SIENTO_QUE_CADA_VEZ_DEPENDO_MAS_DE_LA_IA",
    "SI_NO_TUVIERA_ACCESO_A_IA_MI_RENDIMIENTO_BAJARIA",
    "PREFIERO_USAR_IA_QUE_INVESTIGAR_EN_LIBROS_O_ARTICULOS",
    "CONSIDERO_QUE_PODRIA_DEJAR_DE_USAR_IA_SIN_PROBLEMA",
    "EN_UNA_ESCALA_DEL_1_AL_5_QUE_TAN_DEPENDIENTE_TE_CONSIDERAS_DE_LA_IA_DONDE_1_ES_NADA_DEPENDIENTE_Y_5_ES_TOTALMENTE_DEPENDIENTE"
]

# Solo usa columnas que realmente existen
cols_texto = [c for c in cols_texto_esperadas if c in df.columns]

for col in cols_texto:
    df[col] = df[col].apply(limpiar_texto)
    


# Estandarizaciones puntuales
if "CUAL_ES_EL_USO_PRINCIPAL_QUE_LE_DAS_A_LA_INTELIGENCIA_ARTIFICIAL_QUE_SELECCIONASTE" in df.columns:
    df["CUAL_ES_EL_USO_PRINCIPAL_QUE_LE_DAS_A_LA_INTELIGENCIA_ARTIFICIAL_QUE_SELECCIONASTE"] = df["CUAL_ES_EL_USO_PRINCIPAL_QUE_LE_DAS_A_LA_INTELIGENCIA_ARTIFICIAL_QUE_SELECCIONASTE"].replace({
        "MODO AGENTE" : "ESPECIALIZADO",
        "USO GENERAL" : "GENERAL",
        "OTRO" : "BASICO"
    })
    

if "CON_QUE_FRECUENCIA_USAS_IA_PARA_TAREAS_ESCOLARES" in df.columns:
    df["CON_QUE_FRECUENCIA_USAS_IA_PARA_TAREAS_ESCOLARES"] = df["CON_QUE_FRECUENCIA_USAS_IA_PARA_TAREAS_ESCOLARES"].replace({
        "A VECES" : "POCO FRECUENTE",
        "RARA VEZ" : "POCO FRECUENTE",
        "SIEMPRE" : "CONSTANTEMENTE",
        "CASI SIEMPRE" : "CONSTANTEMENTE",
        "NUNCA" : "NUNCA"
    })
    

if "EN_UNA_ESCALA_DEL_1_AL_5_QUE_TAN_DEPENDIENTE_TE_CONSIDERAS_DE_LA_IA_DONDE_1_ES_NADA_DEPENDIENTE_Y_5_ES_TOTALMENTE_DEPENDIENTE" in df.columns:
    df["EN_UNA_ESCALA_DEL_1_AL_5_QUE_TAN_DEPENDIENTE_TE_CONSIDERAS_DE_LA_IA_DONDE_1_ES_NADA_DEPENDIENTE_Y_5_ES_TOTALMENTE_DEPENDIENTE"] = df["EN_UNA_ESCALA_DEL_1_AL_5_QUE_TAN_DEPENDIENTE_TE_CONSIDERAS_DE_LA_IA_DONDE_1_ES_NADA_DEPENDIENTE_Y_5_ES_TOTALMENTE_DEPENDIENTE"].replace({
        "1.0" : "POCO DEPENDIENTE",
        "2.0" : "POCO DEPENDIENTE",
        "3.0" : "MEDIANAMENTE DEPENDIENTE",
        "4.0" : "ALTAMENTE DEPENDIENTE",
        "5.0" : "ALTAMENTE DEPENDIENTE",
        "NaN" : "DESCONOCIDO"
    })
    
print("Se limpio")
print(df["MARCA_TEMPORAL"].value_counts(dropna=False))

Se limpio
MARCA_TEMPORAL
2026-02-27 21:20:39.590000    1
2026-02-27 21:20:44.168000    1
2026-02-27 21:21:05.880000    1
2026-02-27 21:21:50.365000    1
2026-02-27 21:21:56.632000    1
                             ..
2026-04-13 20:41:44.965000    1
2026-04-13 21:49:23.826000    1
2026-04-13 21:57:05.887000    1
2026-04-15 18:19:46.763000    1
2026-04-16 18:14:12.550000    1
Name: count, Length: 66, dtype: int64


In [ ]:
#BLOQUE 5. Creacion de columnas en df
#Fase KDD: Preprocesamiento

In [ ]:
def extraer_fecha(cadena: str) -> str:
    return pd.Timestamp(cadena).strftime("%Y-%m-%d")

def extraer_hora(cadena: str) -> str:
    return pd.Timestamp(cadena).strftime("%H:%M:%S")

df["FECHA"] = df["MARCA_TEMPORAL"].apply(extraer_fecha)
df["HORA"] = df["MARCA_TEMPORAL"].apply(extraer_hora)

df["CARRERA"] = df["CARRERA"]
df["SEMESTRE"] = df["SEMESTRE_QUE_CURSAS"]
df["EDAD"] = df["EDAD"]
df["USO_PRINCIPAL_IA"] = df["CUAL_ES_EL_USO_PRINCIPAL_QUE_LE_DAS_A_LA_INTELIGENCIA_ARTIFICIAL_QUE_SELECCIONASTE"]
df["FRECUENCIA_USO_IA"] = df["CON_QUE_FRECUENCIA_USAS_IA_PARA_TAREAS_ESCOLARES"]
df["INTENTO_RESOLVER_POR_MI_CUENTA"] = df["INTENTO_RESOLVER_LA_TAREA_POR_MI_CUENTA_ANTES_DE_USAR_IA"]
df["USO_IA_SIN_ENTENDER_CLASE"] = df["USO_IA_CUANDO_NO_ENTIENDO_UN_TEMA_EXPLICADO_EN_CLASE"]
df["VALIDACION_CONTENIDO_IA"] = df["CUANDO_REALIZAS_TRABAJOS_ESCOLARES_CON_APOYO_DE_LA_IA_REVISAS_Y_MODIFICAS_LO_QUE_GENERA_LA_IA_ANTES_DE_ENTREGARLO"]
df["MODELO_IA"] = df["CUAL_ES_LA_INTELIGENCIA_ARTIFICIAL_QUE_USAS_EN_TU_DIA_A_DIA"]
df["SIENTO_QUE_CADA_VEZ_DEPENDO_MAS_DE_LA_IA"] = df["SIENTO_QUE_CADA_VEZ_DEPENDO_MAS_DE_LA_IA"]
df["SI_NO_TUVIERA_ACCESO_A_IA_MI_RENDIMIENTO_BAJARIA"] = df["SI_NO_TUVIERA_ACCESO_A_IA_MI_RENDIMIENTO_BAJARIA"]
df["PREFIERO_IA_SOBRE_ARTICULOS"] = df["PREFIERO_USAR_IA_QUE_INVESTIGAR_EN_LIBROS_O_ARTICULOS"]
df["CONSIDERO_QUE_PODRIA_DEJAR_DE_USAR_IA_SIN_PROBLEMA"] = df["CONSIDERO_QUE_PODRIA_DEJAR_DE_USAR_IA_SIN_PROBLEMA"]
df["DEPENDENCIA_IA"] = df["EN_UNA_ESCALA_DEL_1_AL_5_QUE_TAN_DEPENDIENTE_TE_CONSIDERAS_DE_LA_IA_DONDE_1_ES_NADA_DEPENDIENTE_Y_5_ES_TOTALMENTE_DEPENDIENTE"]





In [11]:
#BLOQUE 6. Variables derivadas
#Fase KDD: Transformación

In [53]:


print(df[[
    "FECHA",
    "HORA",
    "CARRERA",
    "SEMESTRE",
    "EDAD",
    "USO_PRINCIPAL_IA",
    "FRECUENCIA_USO_IA",
    "INTENTO_RESOLVER_POR_MI_CUENTA",
    "USO_IA_SIN_ENTENDER_CLASE",
    "VALIDACION_CONTENIDO_IA",
    "MODELO_IA",
    "SIENTO_QUE_CADA_VEZ_DEPENDO_MAS_DE_LA_IA",
    "SI_NO_TUVIERA_ACCESO_A_IA_MI_RENDIMIENTO_BAJARIA",
    "PREFIERO_IA_SOBRE_ARTICULOS",
    "CONSIDERO_QUE_PODRIA_DEJAR_DE_USAR_IA_SIN_PROBLEMA",
    "DEPENDENCIA_IA"
]].head())

        FECHA      HORA                              CARRERA SEMESTRE  \
0  2026-02-27  21:20:39  INGENIERIA EN SISTEMAS INTELIGENTES        6   
1  2026-02-27  21:20:44  INGENIERIA EN SISTEMAS INTELIGENTES        6   
2  2026-02-27  21:21:05  INGENIERIA EN SISTEMAS INTELIGENTES        6   
3  2026-02-27  21:21:50  INGENIERIA EN SISTEMAS INTELIGENTES        6   
4  2026-02-27  21:21:56  INGENIERIA EN SISTEMAS INTELIGENTES        6   

           EDAD USO_PRINCIPAL_IA FRECUENCIA_USO_IA  \
0  MAYOR QUE 20          GENERAL    POCO FRECUENTE   
1  MAYOR QUE 20          GENERAL    CONSTANTEMENTE   
2  MAYOR QUE 20    ESPECIALIZADO    POCO FRECUENTE   
3  MAYOR QUE 20          GENERAL    CONSTANTEMENTE   
4  MAYOR QUE 20          GENERAL    CONSTANTEMENTE   

  INTENTO_RESOLVER_POR_MI_CUENTA USO_IA_SIN_ENTENDER_CLASE  \
0                        A VECES                   A VECES   
1                        A VECES                   A VECES   
2                        SIEMPRE                  

In [13]:
#BLOQUE 7. Guardar base limpia
#Fase KDD: Cierre del preprocesamiento

In [54]:
df.to_excel("usoia_limpia.xlsx", index=False)
print("Archivo guardado: usoia_limpia.xlsx")

Archivo guardado: usoia_limpia.xlsx
